# Feature Scaling

**DS4DH Practice Pack · Module 06 — Clustering and Segmentation**

*Technique:* StandardScaler, and why distance-based methods need it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/06a_feature_scaling.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Clustering measures distance between places. Distance sums squared differences
across every feature — so a feature measured in hundreds of thousands contributes
a million times more than one measured in single digits.

Scaling is not a tidying step before the real work. It **is** part of the model,
and skipping it silently means clustering on one variable.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
# The scale mismatch, before anything is scaled.
print(f'{"Feature":<20}{"min":>14}{"mean":>14}{"max":>14}{"sd":>14}')
print('-' * 76)
for c in RAW_NEEDED:
    s = feat[c]
    print(f'{c:<20}{s.min():>14,.1f}{s.mean():>14,.1f}{s.max():>14,.1f}{s.std():>14,.1f}')

`tot_income` runs into the hundreds of thousands and `tot_pop` into the millions,
while `Total` and `renter_owner_gap` are single- and double-digit percentages.

`tot_pop` also spans four orders of magnitude, which is why it enters the feature
set as `log10`. Standardising a variable that skewed still leaves one CSD sitting
many standard deviations from everything else, dominating the geometry.

In [ ]:
print(f'tot_pop      skew: {feat["tot_pop"].skew():>8.2f}')
print(f'log10(pop)   skew: {feat["log_pop"].skew():>8.2f}')
print()
print('Log first, then standardise. Standardising alone does not fix skew —')
print('it only recentres and rescales it.')

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(feat[FEATURES])
X_df = pd.DataFrame(X, columns=FEATURES, index=feat.index)

print('After standardising — every feature has mean 0 and sd 1:')
print()
print(f'{"Feature":<20}{"mean":>10}{"sd":>10}{"min":>10}{"max":>10}')
print('-' * 60)
for c in FEATURES:
    s = X_df[c]
    print(f'{c:<20}{s.mean():>10.3f}{s.std():>10.3f}{s.min():>10.2f}{s.max():>10.2f}')

## What changes in the distances

The demonstration: compute the distance between two specific CSDs before and
after scaling, and see which feature drove it each time.

In [ ]:
a, b = feat.index[0], feat.index[1]

print(f'{"Feature":<20}{"raw diff":>16}{"raw contrib":>15}{"scaled contrib":>17}')
print('-' * 68)
raw_sq, sc_sq = 0.0, 0.0
for c in FEATURES:
    rd = feat.loc[a, c] - feat.loc[b, c]
    sd = X_df.loc[a, c] - X_df.loc[b, c]
    raw_sq += rd ** 2
    sc_sq += sd ** 2
    print(f'{c:<20}{rd:>16,.2f}{rd ** 2:>15,.1f}{sd ** 2:>17.3f}')
print('-' * 68)
print(f'{"total squared dist":<20}{"":>16}{raw_sq:>15,.1f}{sc_sq:>17.3f}')
print()
print(f'Unscaled, tot_income accounts for {(feat.loc[a, "tot_income"] - feat.loc[b, "tot_income"]) ** 2 / raw_sq:.1%}')
print('of the distance between these two places. Everything else is rounding.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].boxplot([feat[c] for c in FEATURES])
axes[0].set_xticklabels(FEATURES, rotation=20, ha='right')
axes[0].set_yscale('log')
axes[0].set_title('Raw features (log y-axis — they do not share a scale)')

axes[1].boxplot([X_df[c] for c in FEATURES])
axes[1].set_xticklabels(FEATURES, rotation=20, ha='right')
axes[1].axhline(0, color='#E8663D', lw=1)
axes[1].set_title('After StandardScaler — comparable')
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Remove `tot_income` from `FEATURES`, re-run the scaling and the distance
breakdown.

Which feature dominates the unscaled distance now? The lesson is that the
dominant feature is whichever happens to have the largest units — an arbitrary
fact about measurement, not about housing.

### 🔧 Your turn 2

Try `MinMaxScaler` instead of `StandardScaler`:

```python
from sklearn.preprocessing import MinMaxScaler
X2 = MinMaxScaler().fit_transform(feat[FEATURES])
```

Both put features on a comparable footing. How do they differ in how they treat
an outlier? Which would you use if one CSD had a wildly extreme income?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With income removed, `tot_pop` dominates — and if you drop that
too, `Total` and `renter_owner_gap` finally compete on similar terms because both
are percentages. The general point: without scaling, your clustering is
determined by your units. Report income in thousands rather than dollars and you
get a different answer to the same question, which is not a property any method
should have.

**Your turn 2.** StandardScaler centres on the mean and divides by the standard
deviation, both of which an outlier distorts. MinMaxScaler maps to [0, 1] using
the min and max, so a single extreme value compresses every other observation into
a narrow band. With a wildly extreme income, neither is good — `RobustScaler`
(median and IQR) is the right tool, or transform the variable first, which is
exactly what `log_pop` does here.

</details>

## Where this stops

The feature matrix is now geometrically honest. Nothing has been clustered yet —
that is the next notebook.